In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "ventas")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_sales_stores_transformed = spark.table(f"{catalogo}.{esquema_source}.sales_stores_transformed")

In [0]:
df_transformed = df_sales_stores_transformed.groupBy(col("sale_year"),col("product_name")).agg(
                                                     count(col("sale_id")).alias("conteo"),
                                                     max(col("sale_amount")).alias("max_sale_amount"),
                                                     min(col("sale_amount")).alias("min_sale_amount"),
                                                     max(col("country")).alias("country"),
                                                     max(col("sale_type")).alias("sale_type"),
                                                     max(col("high_efficiency")).alias("high_efficiency")
                                                     ).orderBy(col("sale_year").desc(),col("product_name").asc())

In [0]:
df_transformed.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.golden_sales_partitioned")